# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library and the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Install the mlcroissant library if not installed
!pip install mlcroissant -q

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. This ensures you can dynamically explore all the record sets and fields provided.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata overview
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Date Published: {dataset.metadata.datePublished}")
print("--- Keywords ---")
print(dataset.metadata.keywords)
print("--- Authors ---")
print(dataset.metadata.author)

## 2. Data Overview
The Croissant schema organizes data into *record sets* (`@id`), each with a set of fields (columns). Here, we'll list all record sets, their `@id`s, and the fields they contain.

> **Note:** *You must always use the `@id` of entities (record sets, fields) to reference them in further operations.*

In [ ]:
# List all record sets with their @ids and fields
record_sets = list(dataset.record_sets.values())
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    print("  Fields:")
    for field in rs.fields.values():
        print(f"    - Field Name: {field.name} | @id: {field.id} | DataType: {getattr(field, 'dataType', None)}")
    print("-")

## 3. Data Extraction
Extract the full contents of a record set into a DataFrame. We'll demonstrate with the main record set, which is usually the primary survey table.

First, select the record set's `@id` you wish to analyze further. _(You may review the list above; we'll pick the first as an example.)_

In [ ]:
# Extract all record set @ids
record_set_ids = [rs.id for rs in record_sets]
print("Record set @ids:")
for rid in record_set_ids:
    print(f"- {rid}")

# Select the main record set for survey data (typically the first one, adjust as needed)
main_record_set_id = record_set_ids[0]

# Load records into a pandas DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)

print(f"Columns in record set ({main_record_set_id}):\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Now, let's process the data: filter, normalize, and group using the field `@id`s from our record set.

First, list all column names and choose a numeric field (e.g., GAD-7, PHQ-9, Age) and a potential group (e.g., gender, village).

In [ ]:
# List all columns
print("DataFrame columns:")
for col in df.columns:
    print(f"- {col}")

# Choose a numeric field and a group field by their @ids. Adjust as necessary for your dataset.
# Example: 'gad7_score', 'gender', 'age', 'village', etc.
# These IDs must match the column names as they appear in your DataFrame.
# We'll attempt to auto-detect likely candidates for numeric and group fields.
import re

numeric_candidates = [col for col in df.columns if re.search(r'gad7|phq9|psq|score|age', col, re.IGNORECASE)]
print(f"Numeric candidate columns: {numeric_candidates}")
group_candidates = [col for col in df.columns if re.search(r'gender|village|education|marital|group', col, re.IGNORECASE)]
print(f"Group candidate columns: {group_candidates}")

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    numeric_field_id = df.columns[0]

if group_candidates:
    group_field_id = group_candidates[0]
else:
    group_field_id = df.columns[1]

print(f"Selected numeric field (@id): {numeric_field_id}")
print(f"Selected group field (@id): {group_field_id}")

In [ ]:
threshold = df[numeric_field_id].median() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else None

# Filtering for numeric values > threshold (if field exists and is numeric)
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id, group_field_id]].head())

    # Add normalized version
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized", group_field_id]].head())

    # Group by categorical/group field, if present
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df)
else:
    print(f"The field '{numeric_field_id}' is not numeric or not present. Please select a different numeric field.")

## 5. Visualization
Let's visualize the selected numeric field's distribution and how it varies by the chosen group field, if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Boxplot by group field (if both are present and group is categorical)
if pd.api.types.is_categorical_dtype(df[group_field_id]) or df[group_field_id].dtype == 'object':
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

We have:
- Loaded the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using the Croissant schema and `mlcroissant`.
- Examined available record sets and field `@id`s for detailed, reproducible data access.
- Extracted and analyzed a main record set; filtered and normalized a key numeric field.
- Visualized field distributions and observed group-wise patterns using field `@id`s.

Explore further by adjusting the field/group selectors to examine more variables, or consult metadata for other available record sets in this rich and AI-ready dataset.